# F5 homework: simulation and estimation

Four small studies that use the main tools of F5: estimate a probability by **simulation**, put an error bar on an
estimate with the **bootstrap**, decide whether an A/B difference is real with a **permutation test**, and score a
toy classifier with **cross-entropy**.

**How to work:** fill in each `TODO` cell, then run the check cell below it. A check prints `OK` when your code is
right. Restart and run all cells at the end to make sure everything passes in order. Some tasks also ask for a short
written answer in a markdown cell: those are part of the rubric.

Runs on any laptop or on free Colab (CPU). Only NumPy, matplotlib and SciPy are needed, and every cell finishes in a
few seconds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(0)

## Task 1: the birthday problem by simulation

In a room of $k$ people, what is the chance that at least two share a birthday? Assume 365 equally likely birthdays
and ignore leap years.

1. Write `p_shared_birthday(k, n_sim, rng)` that simulates `n_sim` rooms, each with `k` birthdays drawn by
   `rng.integers(0, 365, size=(n_sim, k))`, and returns the fraction of rooms with at least one shared birthday.
   Hint: sort each row with `np.sort(..., axis=1)`; a shared birthday means two **neighbouring** sorted values are
   equal, which `np.diff(..., axis=1) == 0` detects.
2. Write `p_shared_exact(k)` with the complement rule: the probability that all $k$ birthdays differ is
   $\frac{365}{365} \times \frac{364}{365} \times \dots \times \frac{365 - k + 1}{365}$.

In [ ]:
def p_shared_birthday(k, n_sim, rng):
    # TODO
    pass

def p_shared_exact(k):
    # TODO
    pass

In [ ]:
assert p_shared_exact(23) is not None and p_shared_birthday(5, 10, rng) is not None, "Both functions should return a number"
assert abs(p_shared_exact(23) - 0.5072972) < 1e-6, "With 23 people the exact answer is about 0.5073"
assert abs(p_shared_exact(1)) < 1e-12, "One person cannot share a birthday"
for _k in [10, 23, 40]:
    _sim = p_shared_birthday(_k, 100_000, np.random.default_rng(_k))
    assert abs(_sim - p_shared_exact(_k)) < 0.01, f"For k = {_k} the simulation should be within 0.01 of the exact answer"
print("OK")

Now see how the simulation error shrinks. For `n_sim` in `[100, 1_000, 10_000, 100_000]`, run the $k = 23$
simulation with **20 different seeds** each and record the standard deviation of the 20 estimates in `spread`
(a list of 4 numbers). Then plot `spread` against `n_sim` on log-log axes (`ax.loglog`) together with the theoretical
standard error $\sqrt{p(1 - p)/n}$, where $p$ is the exact answer.

In [ ]:
n_sims = [100, 1_000, 10_000, 100_000]
spread = []
# TODO: fill spread, then make the log-log plot

In [ ]:
assert len(spread) == 4, "spread should hold one number per value of n_sim"
_p = p_shared_exact(23)
for _n, _s in zip(n_sims, spread):
    _se = np.sqrt(_p * (1 - _p) / _n)
    assert 0.5 * _se < _s < 1.6 * _se, f"For n_sim = {_n} the spread should be near sqrt(p(1-p)/n) = {_se:.4f}, got {_s:.4f}"
print("OK")

**Write:** how many times more simulations does it take to make the estimate 10 times more precise, and how does
your plot show it?

*Your answer here.*

## Task 2: a bootstrap confidence interval for a mean

A food-delivery company measures 60 delivery times (minutes). Delivery times are right-skewed: most are quick, a few
are very slow.

In [ ]:
delivery = np.round(np.random.default_rng(5).gamma(shape=2.0, scale=15.0, size=60), 1)
print(delivery[:10], delivery.mean().round(2), np.median(delivery))

1. Write `normal_ci(x)`: the approximate 95% interval $\bar{x} \pm 1.96\,s/\sqrt{n}$ with `ddof=1`.
2. Write `bootstrap_ci(x, stat, n_boot=10_000, seed=0)`: the 95% percentile bootstrap interval. Draw all resample
   indices at once with `rng.integers(0, len(x), size=(n_boot, len(x)))`, compute `stat(x[idx], axis=1)`, and return
   the 2.5th and 97.5th percentiles.
3. Compute `ci_mean_normal`, `ci_mean_boot` and `ci_median_boot` for `delivery`.

In [ ]:
def normal_ci(x):
    # TODO
    pass

def bootstrap_ci(x, stat, n_boot=10_000, seed=0):
    # TODO
    pass

ci_mean_normal = None   # TODO
ci_mean_boot = None     # TODO
ci_median_boot = None   # TODO

In [ ]:
assert None not in (ci_mean_normal, ci_mean_boot, ci_median_boot), "Compute all three intervals"
_se = delivery.std(ddof=1) / np.sqrt(len(delivery))
assert np.allclose(ci_mean_normal, (delivery.mean() - 1.96 * _se, delivery.mean() + 1.96 * _se)), "normal_ci should be mean +- 1.96 * s / sqrt(n) with ddof=1"
assert ci_mean_boot[0] < delivery.mean() < ci_mean_boot[1], "The bootstrap CI should contain the sample mean"
assert abs(ci_mean_boot[0] - ci_mean_normal[0]) < 1.5 and abs(ci_mean_boot[1] - ci_mean_normal[1]) < 1.5, "For 60 values the two intervals for the mean should be close"
assert ci_median_boot[0] < np.median(delivery) < ci_median_boot[1], "The median CI should contain the sample median"
_w30 = np.subtract(*bootstrap_ci(delivery[:30], np.mean)[::-1])
_w60 = np.subtract(*ci_mean_boot[::-1])
assert _w60 < _w30, "More data should give a narrower interval"
assert bootstrap_ci(delivery, np.mean, seed=3) == bootstrap_ci(delivery, np.mean, seed=3), "Use the seed argument"
print("OK")

**Check the method.** We generated the data from a gamma distribution whose true mean is $2 \times 15 = 30$ minutes.
Repeat the whole study 1,000 times: each time draw a fresh sample of 60 with
`rng.gamma(shape=2.0, scale=15.0, size=60)`, build `normal_ci`, and count how often it contains 30. Store the fraction
in `coverage`. (The bootstrap version works the same way but is slower; try it with `n_boot=2000` if you like.)

In [ ]:
coverage = None   # TODO

In [ ]:
assert coverage is not None and 0.88 < coverage < 0.98, f"About 93 to 95 percent of intervals should contain 30, got {coverage}"
print("OK")

**Write:** in one or two sentences, what does "95% confidence" mean here, and why might the coverage come out a
little below 95%?

*Your answer here.*

## Task 3: an A/B test with a permutation test

An online shop tested a new checkout page. Visitors were assigned at random: 1,200 saw the old page (A) and 1,150 the
new one (B). Each entry is 1 if the visitor bought something.

In [ ]:
_ab = np.random.default_rng(21)
conv_a = (_ab.random(1200) < 0.080).astype(int)
conv_b = (_ab.random(1150) < 0.110).astype(int)
print(conv_a.mean().round(4), conv_b.mean().round(4))

1. Write `perm_test(a, b, n_perm=10_000, seed=0)` returning `(observed, p_value)`, where `observed` is
   `mean(b) - mean(a)` and `p_value` is the **two-sided** permutation p-value: shuffle the pooled data, split at
   `len(a)`, and count shuffled differences with absolute value at least `abs(observed)`.
2. Run it on the shop data and store the results in `observed` and `p_ab`.
3. Plot a histogram of the shuffled differences with a vertical line (`ax.axvline`) at the observed difference.
   (You may return the shuffled differences too, or recompute them, for the plot.)
4. Put an error bar on the effect: a 95% bootstrap CI for the difference in conversion rates, `ci_diff`. Resample A
   and B **separately**, each with replacement and at its own size, and take percentiles of the differences.

In [ ]:
def perm_test(a, b, n_perm=10_000, seed=0):
    # TODO
    pass

observed, p_ab = None, None   # TODO
ci_diff = None                # TODO

In [ ]:
assert observed is not None and p_ab is not None, "Run perm_test on the shop data and store observed and p_ab"
assert abs(observed - (conv_b.mean() - conv_a.mean())) < 1e-12, "observed should be mean(b) - mean(a)"
assert 0.0 < p_ab < 0.1, f"For this data the two-sided p-value is a few percent, got {p_ab}"
_o, _p = perm_test(np.array([0, 1, 0, 1]), np.array([1, 0, 1, 0]))[:2]
assert _o == 0 and _p == 1.0, "Equal means: every shuffled difference is at least 0 in size, so p = 1"
_o, _p = perm_test(np.zeros(200), np.ones(200), n_perm=500)[:2]
assert _p == 0.0, "Completely separated groups should give p = 0"
_o1, _p1 = perm_test(conv_a, conv_b, seed=1)[:2]
_o2, _p2 = perm_test(conv_b, conv_a, seed=2)[:2]
assert abs(_p1 - _p2) < 0.02, "A two-sided test should give about the same p-value when the groups are swapped"
assert ci_diff is not None and ci_diff[0] < observed < ci_diff[1], "The CI for the difference should contain the observed difference"
assert 0.02 < ci_diff[1] - ci_diff[0] < 0.07, "The CI width should be a few percentage points"
print("OK")

**Write:** report the result the way you would to the shop owner, in two or three sentences: the observed
difference, its confidence interval, the p-value, and what the p-value does **not** tell them.

*Your answer here.*

## Task 4: cross-entropy of a toy classifier

A classifier sorts 8 photos into 3 classes (0 = cat, 1 = dog, 2 = rabbit). Below are its raw scores (logits) and the
true labels.

In [ ]:
logits = np.array([[ 2.0,  0.5, -1.0],
                   [ 0.2,  1.8,  0.1],
                   [-0.5,  0.0,  2.5],
                   [ 1.1,  1.0,  0.9],
                   [ 3.0, -1.0, -1.0],
                   [ 0.1,  0.3,  2.0],
                   [ 2.2,  0.1,  0.0],
                   [-1.0,  2.5,  0.5]])
labels = np.array([0, 1, 2, 1, 0, 1, 0, 1])

1. Write a numerically stable `softmax(z)` that works **row by row** on a 2-D array (subtract each row's maximum,
   `z.max(axis=1, keepdims=True)`).
2. Write `ce_loss(probs, labels, eps=1e-12)`: the average of $-\ln q_{i, y_i}$, clipping the picked probabilities to
   `[eps, 1]`.
3. Compute `probs`, `loss`, `accuracy` (the fraction of rows whose largest probability is on the true class) and
   `baseline` $= \ln 3$, the loss of a uniform guess.
4. Row 3 (true class 1) is already wrong, but only just: the model slightly prefers class 0. Make it **confidently**
   wrong: copy `logits`, set row 3 to `[0.0, -6.0, 6.0]`, and recompute the loss as `loss_bad` and the accuracy as
   `accuracy_bad`.

In [ ]:
def softmax(z):
    # TODO
    pass

def ce_loss(probs, labels, eps=1e-12):
    # TODO
    pass

probs = None      # TODO
loss = None       # TODO
accuracy = None   # TODO
baseline = None   # TODO
loss_bad, accuracy_bad = None, None   # TODO

In [ ]:
assert probs is not None and probs.shape == (8, 3), "probs should be an 8 x 3 array"
assert np.allclose(probs.sum(axis=1), 1), "Each row of probs should add up to 1"
assert np.allclose(softmax(np.array([[1000.0, 999.0, 998.0]])), softmax(np.array([[2.0, 1.0, 0.0]]))), "softmax should not overflow for large logits"
_q = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
assert abs(loss - (-np.mean(np.log(_q[np.arange(8), labels])))) < 1e-9, "loss should be the mean of -ln(probability of the true class)"
assert baseline is not None and loss_bad is not None, "Compute baseline, loss_bad and accuracy_bad too"
assert loss < baseline, "This classifier should beat the uniform guess"
assert abs(baseline - np.log(3)) < 1e-12, "baseline is ln 3"
assert accuracy == 0.75, "6 of the 8 examples have their largest probability on the true class"
assert loss_bad > loss + 1.0 and accuracy_bad == accuracy, "Turning a narrow mistake into a confident one should raise the loss a lot while the accuracy stays the same"
print("OK")

**Write:** the accuracy did not change, but the loss did. Why does cross-entropy punish the confident mistake so much
more than the narrow one, and why is that a useful property for a training loss? (Two or three sentences.)

*Your answer here.*

## Stretch: watch the central limit theorem

Pick a distribution that looks nothing like a bell curve, for example the delivery-time gamma distribution from
Task 2 or `rng.exponential`. For sample sizes $n = 1, 2, 5, 30$, draw 20,000 samples, compute their means, and plot
four histograms side by side with the normal curve $\mathcal{N}(\mu, \sigma^2/n)$ on top
(`stats.norm.pdf(x, mu, sigma / np.sqrt(n))`, with `density=True` in `ax.hist`). At which $n$ does the histogram
start to look normal?

In [ ]:
# Stretch: your code here